# MonIA Kaggle — ONE CLICK
Candidate-only MonIA worker for the new-game opening. Identity refs are validated before generation.


In [ ]:
%pip -q install -U diffusers transformers accelerate safetensors imageio[ffmpeg] av huggingface_hub requests ftfy "pillow==11.3.0"
print('✅ Dependencies ready')


In [ ]:
import os, json, requests, torch, PIL, imageio.v3 as iio, base64, io
from pathlib import Path
from PIL import Image
from diffusers.utils import export_to_video, load_image
print('Pillow:', PIL.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU first')
REPO='vartcom38-collab/marion-lucas-game'
RAW=f'https://raw.githubusercontent.com/{REPO}/main'
WORK=Path('/kaggle/working/monia-studio'); WORK.mkdir(parents=True, exist_ok=True)
print('✅ MonIA one-click ready')


In [ ]:
def get_json(url):
    r=requests.get(url, timeout=30); r.raise_for_status(); return r.json()
def download(url, target):
    r=requests.get(url, timeout=120); r.raise_for_status(); Path(target).write_bytes(r.content); return str(target)
def download_text(url):
    r=requests.get(url, timeout=120); r.raise_for_status(); return r.text.strip()
def decode_b64_text(text):
    payload=''.join(text.split()); payload += '=' * (-len(payload) % 4); return base64.b64decode(payload)
def save_valid_image(raw, target):
    try:
        im=Image.open(io.BytesIO(raw)); im.load(); im=im.convert('RGB')
        Path(target).parent.mkdir(parents=True, exist_ok=True)
        im.save(target, format='JPEG', quality=92)
        return str(target)
    except Exception as exc:
        print('⚠️ Invalid identity image skipped:', target, repr(exc)); return None
def materialize_atlas(ch, target):
    chunks=ch.get('atlasChunks') if isinstance(ch.get('atlasChunks'),list) else []
    if not chunks: return None
    raw=decode_b64_text(''.join(download_text(url) for url in chunks))
    Path(target).write_bytes(raw)
    try:
        im=Image.open(target); im.load(); print('✅ Atlas validated:', im.format, im.size)
    except Exception as exc:
        raise RuntimeError(f'Invalid atlas image: {exc}')
    return str(target)
def materialize_priority_images(ch, target_dir):
    urls=ch.get('priorityImages') if isinstance(ch.get('priorityImages'),list) else []
    out=[]
    for i,url in enumerate(urls, start=1):
        try:
            raw=decode_b64_text(download_text(url))
            target=Path(target_dir)/f'priority-{i:02d}.jpg'
            valid=save_valid_image(raw,target)
            if valid: out.append(valid)
        except Exception as exc:
            print('⚠️ Priority ref decode failed, skipping:', i, repr(exc))
    return out
candidates=[Path('job.json'),Path('/kaggle/working/job.json')]
input_root=Path('/kaggle/input')
if input_root.exists(): candidates.extend(input_root.rglob('job.json'))
bundle_job=next((p for p in candidates if p.exists()),None)
job=json.loads(bundle_job.read_text(encoding='utf-8')) if bundle_job else get_json(RAW + '/studio/queue/test-kaggle-marion-001.json')
print('✅ Dynamic gameplay job loaded' if bundle_job else 'ℹ️ Repository test job loaded')
assert job.get('candidateOnly') is True and job.get('narrativeAuthority') is False
job_id=job['id']; job_dir=WORK/job_id; job_dir.mkdir(parents=True, exist_ok=True)
refs={}; ref_packs={}
for ch in job.get('characters', []):
    cid=ch['id']; priority=materialize_priority_images(ch, job_dir/f'pack-{cid}')
    if priority:
        ref_packs[cid]=priority; refs[cid]=priority[0]; print('✅ Valid priority identity pack:', cid, len(priority)); continue
    atlas=materialize_atlas(ch, job_dir/f'canon-{cid}.webp')
    if atlas:
        refs[cid]=atlas; ref_packs[cid]=[atlas]; print('✅ Canon atlas materialized:', cid); continue
    if ch.get('canonRef'):
        fallback_path=job_dir/f'canon-{cid}-fallback.jpg'; raw=requests.get(ch['canonRef'],timeout=120); raw.raise_for_status()
        fallback=save_valid_image(raw.content,fallback_path)
        if fallback:
            refs[cid]=fallback; ref_packs[cid]=[fallback]; print('ℹ️ Canon fallback image validated:', cid); continue
    raise RuntimeError(f'No valid canonical identity source for {cid}')
print('✅ Job loaded:', job_id, 'refs:', list(refs))


In [ ]:
from diffusers import LTXImageToVideoPipeline
g=job.get('generation') or {}
print('⏳ Loading LTX model...')
pipe=LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
primary=job.get('primaryCharacter') or ('lucas' if 'lucas' in refs else 'marion' if 'marion' in refs else next(iter(refs.keys())))
base_seed=int(g.get('seed',240907))
shot_characters=g.get('shotCharacters') if isinstance(g.get('shotCharacters'),list) else []
shot_images=g.get('shotImages') if isinstance(g.get('shotImages'),list) else []
shot_prompts=g.get('shotPrompts') if isinstance(g.get('shotPrompts'),list) else []
shot_reference_index=g.get('shotReferenceIndex') if isinstance(g.get('shotReferenceIndex'),list) else []
shot_count=max(1,min(6,int(g.get('shotCount') or max(2,len(shot_prompts),len(shot_characters),len(shot_images)))))
clips=[]; continuity_list=[]; previous_image=None
for index in range(1,shot_count+1):
    seed=base_seed+index-1; name=f'shot-{index:02d}.mp4'
    requested=shot_characters[index-1] if index-1 < len(shot_characters) else None
    image_url=shot_images[index-1] if index-1 < len(shot_images) else None
    ref_index=shot_reference_index[index-1] if index-1 < len(shot_reference_index) and isinstance(shot_reference_index[index-1],int) else 0
    if image_url:
        custom=download(image_url, job_dir/f'shot-source-{index:02d}.jpg'); image=load_image(custom); source='generated-image'
    elif requested in ref_packs:
        pack=ref_packs[requested]; chosen=pack[max(0,min(ref_index,len(pack)-1))]; image=load_image(chosen); source='canonical-priority-pack' if len(pack)>1 else 'canonical-pack'
    elif previous_image is not None:
        image=previous_image; source='previous-video-frame'
    else:
        image=load_image(refs[primary]); source='canonical-pack'
    prompt=shot_prompts[index-1] if index-1 < len(shot_prompts) and shot_prompts[index-1] else job['prompt']
    print(f'🎬 Generating candidate {index}/{shot_count} from {source} ({requested or image_url or primary})...')
    generator=torch.Generator(device='cpu').manual_seed(seed)
    frames=pipe(image=image,prompt=prompt,negative_prompt=job.get('negativePrompt'),width=int(g.get('width',768)),height=int(g.get('height',432)),num_frames=int(g.get('frames',49)),num_inference_steps=int(g.get('steps',8)),generator=generator).frames[0]
    out=str(job_dir/name); export_to_video(frames,out,fps=int(g.get('fps',12)))
    clips.append(name); continuity_list.append(source); print('✅ Candidate generated:',out)
    last=iio.imread(out,index=-1,plugin='pyav'); frame_path=job_dir/f'continuity-shot-{index:02d}.png'; iio.imwrite(frame_path,last); previous_image=load_image(str(frame_path))


In [ ]:
result={'jobId':job_id,'state':'candidate','candidateOnly':True,'narrativeAuthority':False,'router':'ltx','selectionMode':'surprise-auto','continuity':continuity_list,'clips':clips,'shotCount':len(clips)}
(job_dir/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print('✅ Candidate package ready for GitHub Actions pickup')
